In [9]:
import pandas as pd
import glob
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm   # ✅ for realtime progress bar
import os
os.chdir("/tmp")
# Define the paths
digenic_file = "helper_files/specific_digenic_combination.csv"
input_folder = "Individual_data_Uniti_generated_features_new"
#input_folder = '/home/mdnl/myRDS/PabloRDS/Hamza/Individual_data_Uniti_generated_features_cohort'
output_folder = "Individual_data_Uniti_generated_features_withDCs_new"

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# ---------------------------#
#    LOAD DIGENIC DATA       #
# ---------------------------#

df_digenic = pd.read_csv(digenic_file)

# Build dictionary of protein → max combined_score
protein_scores = {}
for _, row in df_digenic.iterrows():
    p1, p2, score = row["protein1_original_name"], row["protein2_original_name"], row["combined_score"]
    protein_scores[p1] = max(protein_scores.get(p1, 0), score)
    protein_scores[p2] = max(protein_scores.get(p2, 0), score)

# ---------------------------#
#   PROCESSING FUNCTION      #
# ---------------------------#

def process_exome_file(exome_file):
    file_name = os.path.basename(exome_file)
    try:
        df_exome = pd.read_csv(exome_file, sep="\t", low_memory=False)
    except Exception as e:
        return f"❌ Error reading {file_name}: {e}"

    if "SYMBOL" not in df_exome.columns:
        return f"⚠️ Skipping {file_name}: Missing SYMBOL column."

    # Vectorised: map SYMBOL → protein_scores
    df_exome["DCs_score"] = df_exome["SYMBOL"].map(protein_scores).fillna(0).astype(int)

    # Save updated file
    output_file = os.path.join(output_folder, file_name)
    try:
        df_exome.to_csv(output_file, sep="\t", index=False)
        return f"✅ Processed {file_name}"
    except Exception as e:
        return f"❌ Failed to save {file_name}: {e}"

# ---------------------------#
#   PARALLEL FILE PROCESS    #
# ---------------------------#

if __name__ == "__main__":
    exome_files = glob.glob(os.path.join(input_folder, "*.tsv"))

    results = []
    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(process_exome_file, f): f for f in exome_files}

        # tqdm progress bar for realtime feedback
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing files", unit="file"):
            results.append(future.result())

    # Print final per-file results
    for r in results:
        print(r)

    print(f"\n🎉 Processing completed. Updated files are saved in '{output_folder}'.")

Processing files: 100%|████████████████████| 166/166 [00:00<00:00, 168.04file/s]


✅ Processed Uniti_case_ZU69.tsv
✅ Processed Uniti_case_ZU86.tsv
✅ Processed Uniti_case_GR24.tsv
✅ Processed Uniti_case_PT207.tsv
✅ Processed Uniti_case_PT115.tsv
✅ Processed Uniti_case_ZU09.tsv
✅ Processed Uniti_case_ZU54.tsv
✅ Processed Uniti_case_GR2.tsv
✅ Processed Uniti_case_PT105.tsv
✅ Processed Uniti_case_ZU22.tsv
✅ Processed Uniti_case_GR115.tsv
✅ Processed Uniti_case_GR75.tsv
✅ Processed Uniti_case_GR33.tsv
✅ Processed Uniti_case_PT54.tsv
✅ Processed Uniti_case_GR150.tsv
✅ Processed Uniti_case_PT121.tsv
✅ Processed Uniti_case_ZU36.tsv
✅ Processed Uniti_case_PT51.tsv
✅ Processed Uniti_case_GR06.tsv
✅ Processed Uniti_case_GR66.tsv
✅ Processed Uniti_case_PT156.tsv
✅ Processed Uniti_case_GR34.tsv
✅ Processed Uniti_case_ZU29.tsv
✅ Processed Uniti_case_GR23.tsv
✅ Processed Uniti_case_ZU72.tsv
✅ Processed Uniti_case_ZU051.tsv
✅ Processed Uniti_case_GR53.tsv
✅ Processed Uniti_case_PT25.tsv
✅ Processed Uniti_case_ZU59.tsv
✅ Processed Uniti_case_GR114.tsv
✅ Processed Uniti_case_GR49.tsv
